# Invoice PDF Reorder — interactive notebook

Wraps `reorder_invoices.py`. Each scanned page is sent to Claude vision (`claude-opus-4-7`) which returns structured `{supplier, amount, date, id_number}`. Then we fuzzy-match each table row to a scanned page and emit the sorted PDF + CSVs.

**Requires** `ANTHROPIC_API_KEY` in the environment.

Outputs:
- `output_sorted.pdf` — scanned pages reordered to match the table.
- `match_report.csv` — per-row matched page + score + status.
- `extracted_invoices.csv` — flat table of `supplier / amount / date / id_number` per scanned page.

In [ ]:
# Cell 0 — paths and setup
import logging, os, sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import reorder_invoices as ri

TABLE_PDF   = Path('samples/table.pdf')
SCANNED_PDF = Path('samples/scanned_invoices.pdf')
OUT_DIR     = Path('samples/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before running'
print('Table PDF exists:', TABLE_PDF.exists())
print('Scanned PDF exists:', SCANNED_PDF.exists())

In [ ]:
# Cell 1 — parse the table PDF
rows = ri.parse_table_pdf(TABLE_PDF)
rows_df = pd.DataFrame([
    {'index': r.index, 'supplier': r.supplier, 'amount': r.amount, 'date': r.date, 'id_number': r.id_number}
    for r in rows
])
print(f'parsed {len(rows)} rows')
rows_df

In [ ]:
# Cell 2 — extract fields from each scanned page via Claude vision (one API call per page)
pages = ri.extract_pages_with_claude(SCANNED_PDF)
print(f'extracted {len(pages)} pages')
ri.pages_to_dataframe(pages)

In [ ]:
# Cell 3 — match rows to pages
matches = ri.match(rows, pages)
matches_df = ri.matches_to_dataframe(matches)

def _flag(s):
    return ['background-color: #ffe5e5' if v in ('unmatched', 'low_confidence') else '' for v in s]

matches_df.style.apply(_flag, subset=['status'])

In [ ]:
# Cell 4 — write outputs
out_pdf = OUT_DIR / 'output_sorted.pdf'
out_csv = OUT_DIR / 'match_report.csv'
extracted_csv = OUT_DIR / 'extracted_invoices.csv'
ri.write_sorted_pdf(SCANNED_PDF, out_pdf, matches, total_pages=len(pages))
ri.write_report_csv(matches, out_csv)
ri.write_extracted_csv(pages, extracted_csv)
print('Wrote:', out_pdf)
print('Wrote:', out_csv)
print('Wrote:', extracted_csv)

## Single-invoice test

Drop one PDF or image in `samples/` and run the cell below — fields are returned as a one-row DataFrame and exported to `samples/out/single_test.csv`.

In [ ]:
SINGLE = Path('samples/single_invoice.pdf')  # or .jpg / .png
fields = ri.extract_single_file(SINGLE)
df = pd.DataFrame([fields])
df.to_csv(OUT_DIR / 'single_test.csv', index=False, encoding='utf-8-sig')
df